# 02_02 — Preparación cartográfica SER

Este notebook prepara las fuentes cartográficas del Servicio de Estacionamiento Regulado (SER) de Madrid desde `data/raw/cartografia/` hacia `data/interim/cartografia/`.

Las fuentes tratadas son el límite SER, los barrios SER, las bandas de aparcamiento SER y el callejero de viales vigentes. La unidad espacial depende de cada fuente: ámbito regulado, división territorial SER, líneas de bandas reguladas y geometría vial.

La granularidad temporal es estática o corresponde a la versión actual descargada de cada fuente. El notebook no construye dificultad SER, agregaciones horarias, paneles, joins finales, métricas proxy ni modelos.

El callejero se utiliza como base vial neutra para el mapa y para contextualizar las bandas reguladas, condicionado a sus validaciones geométricas y de cobertura. Las decisiones de limpieza se documentan dentro del bloque de cada fuente.

## 0. Configuración inicial

La raíz del repositorio se detecta mediante la existencia de `data_catalog.csv`. Todas las rutas se gestionan de forma relativa al repositorio para evitar dependencias del entorno de ejecución.

Las operaciones espaciales se realizan en ETRS89 / UTM zona 30N, EPSG:25830. Este sistema de referencia permite expresar áreas, longitudes y tolerancias espaciales en metros.

Las fuentes de entrada se localizan bajo `data/raw/cartografia/`. Las salidas limpias se escriben en `data/interim/cartografia/` después de superar los diagnósticos propios de cada fuente, las validaciones internas de esquema, CRS, geometría y atributos, y la comprobación técnica de escritura.

In [1]:
from __future__ import annotations

import tempfile
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import geopandas as gpd
import pandas as pd

try:
    from shapely import make_valid as shapely_make_valid
except ImportError:
    shapely_make_valid = None

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv al recorrer la ruta actual y sus padres. "
        f"Ruta inicial: {current}"
    )


ROOT = find_repo_root()
CATALOG_PATH = ROOT / "data_catalog.csv"
TARGET_CRS = "EPSG:25830"

CARTOGRAPHY_DATASET_IDS = [
    "ser_geoportal_limite_ser",
    "ser_geoportal_barrios_ser",
    "ser_geoportal_bandas_aparcamiento",
    "callejero_viales_vigentes",
]

CARTOGRAPHY_RAW_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser.geojson"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser.geojson"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_bandas_aparcamiento/"
        / "SHP_ZIP.zip"
    ),
    "callejero_viales_vigentes": (
        ROOT
        / "data/raw/cartografia/callejero_viales_vigentes/"
        / "contexto_callejero_viales_vigentes__actual.zip"
    ),
}

CANDIDATE_INTERIM_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser_clean.parquet"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser_clean.parquet"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_bandas_aparcamiento/"
        / "ser_geoportal_bandas_aparcamiento_clean.parquet"
    ),
    "callejero_viales_vigentes": (
        ROOT
        / "data/interim/cartografia/callejero_viales_vigentes/"
        / "callejero_viales_vigentes_clean.parquet"
    ),
}

expected_keys = set(CARTOGRAPHY_DATASET_IDS)
if set(CARTOGRAPHY_RAW_PATHS) != expected_keys:
    raise ValueError(
        "Las claves de CARTOGRAPHY_RAW_PATHS no coinciden con CARTOGRAPHY_DATASET_IDS. "
        f"Observado: {sorted(CARTOGRAPHY_RAW_PATHS)}; esperado: {CARTOGRAPHY_DATASET_IDS}"
    )
if set(CANDIDATE_INTERIM_PATHS) != expected_keys:
    raise ValueError(
        "Las claves de CANDIDATE_INTERIM_PATHS no coinciden con CARTOGRAPHY_DATASET_IDS. "
        f"Observado: {sorted(CANDIDATE_INTERIM_PATHS)}; esperado: {CARTOGRAPHY_DATASET_IDS}"
    )

print(f"ROOT: {ROOT}")
print(f"GeoPandas: {gpd.__version__}")
print(f"CRS objetivo: {TARGET_CRS}")

ROOT: /Users/hugo/TFM_parking_madrid
GeoPandas: 1.1.3
CRS objetivo: EPSG:25830


Si no se detecta la raíz del repositorio o GeoPandas no puede cargarse, el notebook no debe continuar. El CRS objetivo permite realizar de forma coherente las operaciones métricas necesarias para la preparación cartográfica.

Las rutas se definen explícitamente para garantizar la reproducibilidad y la trazabilidad de cada fuente y de sus futuras salidas limpias.

## 1. Fuentes de entrada y salidas previstas

El notebook mantiene un identificador estable para cada una de las cuatro fuentes cartográficas y define de forma explícita sus rutas raw y sus salidas interim previstas.

La separación entre datos raw e interim evita modificar las fuentes originales y permite que cada capa limpia se genere únicamente después de superar sus comprobaciones de estructura, calidad geométrica y utilidad para el TFM.

In [2]:
def _relative_to_root(path: Path) -> str:
    try:
        return path.relative_to(ROOT).as_posix()
    except ValueError:
        return str(path)


def _route_error(dataset_id: str, path: Path, observed: Any, expected: str) -> ValueError:
    return ValueError(
        f"Dataset: {dataset_id}; ruta: {_relative_to_root(path)}; "
        f"valor observado: {observed}; condición esperada: {expected}"
    )


catalog = pd.read_csv(CATALOG_PATH)
required_catalog_columns = {
    "dataset_id",
    "archivo_raw",
    "archivo_interim",
    "notebook_generador",
}
missing_catalog_columns = sorted(required_catalog_columns - set(catalog.columns))
if missing_catalog_columns:
    raise ValueError(
        f"Faltan columnas obligatorias en data_catalog.csv: {missing_catalog_columns}"
    )

catalog_scope = catalog.loc[catalog["dataset_id"].isin(CARTOGRAPHY_DATASET_IDS)].copy()
missing_catalog_ids = [
    dataset_id for dataset_id in CARTOGRAPHY_DATASET_IDS
    if dataset_id not in set(catalog_scope["dataset_id"])
]
if missing_catalog_ids:
    raise ValueError(f"Faltan dataset_id en data_catalog.csv: {missing_catalog_ids}")
if catalog_scope["dataset_id"].duplicated().any():
    duplicated_ids = catalog_scope.loc[catalog_scope["dataset_id"].duplicated(), "dataset_id"].tolist()
    raise ValueError(f"Dataset_id duplicados en data_catalog.csv: {duplicated_ids}")

catalog_scope = (
    catalog_scope
    .assign(_order=pd.Categorical(catalog_scope["dataset_id"], categories=CARTOGRAPHY_DATASET_IDS, ordered=True))
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

route_rows = []
for row in catalog_scope.itertuples(index=False):
    dataset_id = row.dataset_id
    raw_path = CARTOGRAPHY_RAW_PATHS[dataset_id]
    candidate_path = CANDIDATE_INTERIM_PATHS[dataset_id]
    catalog_raw_path = ROOT / row.archivo_raw
    catalog_interim_path = ROOT / row.archivo_interim

    if raw_path != catalog_raw_path:
        raise _route_error(
            dataset_id,
            raw_path,
            _relative_to_root(catalog_raw_path),
            "la ruta raw cartográfica debe coincidir con data_catalog.csv",
        )
    if candidate_path != catalog_interim_path:
        raise _route_error(
            dataset_id,
            candidate_path,
            _relative_to_root(catalog_interim_path),
            "la ruta interim prevista debe coincidir con data_catalog.csv",
        )

    if raw_path.suffix.lower() == ".zip" and raw_path.exists():
        with zipfile.ZipFile(raw_path) as archive:
            internal_files = [name for name in archive.namelist() if not name.endswith("/")]
        n_archivos_internos = len(internal_files)
        n_shapefiles_en_zip = sum(name.lower().endswith(".shp") for name in internal_files)
    elif raw_path.suffix.lower() in {".geojson", ".json"}:
        n_archivos_internos = 1
        n_shapefiles_en_zip = pd.NA
    else:
        n_archivos_internos = pd.NA
        n_shapefiles_en_zip = pd.NA

    route_rows.append(
        {
            "dataset_id": dataset_id,
            "archivo_raw": _relative_to_root(raw_path),
            "raw_existe": raw_path.exists(),
            "n_archivos_internos": n_archivos_internos,
            "n_shapefiles_en_zip": n_shapefiles_en_zip,
            "archivo_interim_previsto": _relative_to_root(candidate_path),
        }
    )

route_contract = pd.DataFrame(route_rows)

for dataset_id, raw_path in CARTOGRAPHY_RAW_PATHS.items():
    if not raw_path.exists():
        raise _route_error(dataset_id, raw_path, raw_path.exists(), "el raw cartográfico debe existir")

raw_path_values = list(CARTOGRAPHY_RAW_PATHS.values())
if len(set(raw_path_values)) != len(raw_path_values):
    raise ValueError(
        "Las rutas raw cartográficas deben ser únicas. "
        f"Valor observado: {len(set(raw_path_values))} rutas únicas; esperado: {len(raw_path_values)}"
    )

candidate_path_values = list(CANDIDATE_INTERIM_PATHS.values())
if len(set(candidate_path_values)) != len(candidate_path_values):
    raise ValueError(
        "Las rutas interim previstas deben ser únicas. "
        f"Valor observado: {len(set(candidate_path_values))} rutas únicas; esperado: {len(candidate_path_values)}"
    )

for dataset_id, raw_path in CARTOGRAPHY_RAW_PATHS.items():
    if raw_path.suffix.lower() == ".zip":
        observed = route_contract.loc[
            route_contract["dataset_id"].eq(dataset_id), "n_shapefiles_en_zip"
        ].iloc[0]
        if observed != 1:
            raise _route_error(dataset_id, raw_path, observed, "el ZIP debe contener exactamente un .shp")

observed_ids = route_contract["dataset_id"].tolist()
if observed_ids != CARTOGRAPHY_DATASET_IDS:
    raise ValueError(
        "Los dataset_id de la tabla no coinciden con CARTOGRAPHY_DATASET_IDS. "
        f"Observado: {observed_ids}; esperado: {CARTOGRAPHY_DATASET_IDS}"
    )

route_contract

,dataset_id,archivo_raw,raw_existe,n_archivos_internos,n_shapefiles_en_zip,archivo_interim_previsto
0,ser_geoportal_limite_ser,data/raw/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser.geojson,True,1,<NA>,data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_c...
1,ser_geoportal_barrios_ser,data/raw/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser.geo...,True,1,<NA>,data/interim/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser...
2,ser_geoportal_bandas_aparcamiento,data/raw/cartografia/ser_geoportal_bandas_aparcamiento/SHP_ZIP.zip,True,5,1,data/interim/cartografia/ser_geoportal_bandas_aparcamiento/ser_geoportal_ban...
3,callejero_viales_vigentes,data/raw/cartografia/callejero_viales_vigentes/contexto_callejero_viales_vig...,True,5,1,data/interim/cartografia/callejero_viales_vigentes/callejero_viales_vigentes...


Deben aparecer exactamente cuatro fuentes cartográficas catalogadas y todos los archivos raw deben estar disponibles. La tabla muestra las rutas raw, el contenido interno de los ZIP, la salida interim prevista y el notebook generador declarado en el catálogo.

Si falta algún raw, una ruta está duplicada o un ZIP no contiene un shapefile inequívoco, el proceso debe detenerse antes de leer geometrías.

## 2. Funciones auxiliares geoespaciales

Se definen funciones generales para mostrar rutas relativas, normalizar nombres de columnas, leer GeoJSON, leer un shapefile contenido en ZIP, asegurar EPSG:25830, resumir estructura y calidad geométrica, limpiar texto y escribir Parquet.

Estas funciones no eliminan registros por sí mismas, no reparan geometrías sin decisión explícita y no deciden columnas finales. Las decisiones específicas se documentan dentro del bloque de cada fuente.

In [3]:
def relpath(path: Path) -> str:
    try:
        return Path(path).relative_to(ROOT).as_posix()
    except ValueError:
        return str(path)


def strip_accents(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value)
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_key(value: Any) -> str:
    text = strip_accents(str(value).strip().lower())
    chars = []
    previous_was_separator = False
    for char in text:
        if char.isalnum():
            chars.append(char)
            previous_was_separator = False
        elif not previous_was_separator:
            chars.append("_")
            previous_was_separator = True
    return "".join(chars).strip("_")


def make_unique_columns(columns: list[Any]) -> list[str]:
    seen: dict[str, int] = {}
    unique_columns = []
    for column in columns:
        base = normalize_key(column)
        if not base:
            base = "column"
        count = seen.get(base, 0)
        unique = base if count == 0 else f"{base}_{count}"
        while unique in seen:
            count += 1
            unique = f"{base}_{count}"
        seen[base] = count + 1
        seen[unique] = 1
        unique_columns.append(unique)
    return unique_columns


def normalize_geo_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(f"Se esperaba un GeoDataFrame; observado: {type(gdf).__name__}")

    geometry_name = gdf.geometry.name
    if geometry_name not in gdf.columns:
        raise ValueError("El GeoDataFrame no tiene una columna geométrica activa presente en sus columnas.")

    non_geometry_columns = [column for column in gdf.columns if column != geometry_name]
    normalized_columns = make_unique_columns(["geometry", *non_geometry_columns])
    attribute_columns = normalized_columns[1:]

    attributes = pd.DataFrame(gdf.drop(columns=[geometry_name])).copy()
    attributes.columns = attribute_columns

    geometry = gpd.GeoSeries(
        gdf.geometry.copy(),
        index=gdf.index,
        crs=gdf.crs,
        name="geometry",
    )

    result = gpd.GeoDataFrame(attributes, geometry=geometry, crs=gdf.crs)
    result = result.set_geometry("geometry")
    return result


def read_geojson(path: Path, dataset_id: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            f"gpd.read_file no devolvió un GeoDataFrame: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            "no existe una geometría activa en el GeoDataFrame leído."
        )
    return gdf


def read_shp_zip(path: Path, dataset_id: str) -> gpd.GeoDataFrame:
    with zipfile.ZipFile(path) as archive:
        shp_members = [name for name in archive.namelist() if name.lower().endswith(".shp")]
        if len(shp_members) != 1:
            raise ValueError(
                f"Dataset: {dataset_id}; ruta: {relpath(path)}; valor observado: {len(shp_members)}; "
                "condición esperada: el ZIP debe contener exactamente un .shp"
            )
        shp_member = shp_members[0]
        with tempfile.TemporaryDirectory() as tmpdir:
            archive.extractall(tmpdir)
            shp_path = Path(tmpdir) / shp_member
            if not shp_path.exists():
                raise FileNotFoundError(
                    f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
                    f"no se localizó el shapefile extraído: {shp_member}"
                )
            gdf = gpd.read_file(shp_path)

    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            f"gpd.read_file no devolvió un GeoDataFrame: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            "no existe una geometría activa en el GeoDataFrame leído."
        )
    return gdf


def ensure_crs_25830(gdf: gpd.GeoDataFrame, dataset_id: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: CRS nulo; "
            "condición esperada: CRS declarado y transformable a EPSG:25830"
        )
    try:
        epsg = gdf.crs.to_epsg()
        if epsg == 25830:
            return gdf.copy()
        return gdf.to_crs(TARGET_CRS)
    except Exception as exc:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: {gdf.crs}; "
            "condición esperada: CRS transformable a EPSG:25830"
        ) from exc


def geometry_quality_summary(gdf: gpd.GeoDataFrame, dataset_id: str) -> dict[str, Any]:
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; se esperaba un GeoDataFrame; observado: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(f"Dataset: {dataset_id}; el GeoDataFrame no tiene geometría activa.")
    if gdf.crs is None:
        raise ValueError(f"Dataset: {dataset_id}; el CRS no está declarado.")

    geometry = gdf.geometry
    non_null_geometry = geometry[geometry.notna()]
    geometry_types = sorted(non_null_geometry.geom_type.dropna().unique().tolist())
    empty_count = int(non_null_geometry.is_empty.sum())
    invalid_count = int((~non_null_geometry.is_valid).sum())

    polygon_mask = non_null_geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    line_mask = non_null_geometry.geom_type.isin(["LineString", "MultiLineString"])
    area_total = non_null_geometry.loc[polygon_mask].area.sum() if polygon_mask.any() else pd.NA
    length_total = non_null_geometry.loc[line_mask].length.sum() if line_mask.any() else pd.NA

    return {
        "dataset_id": dataset_id,
        "n_filas": len(gdf),
        "n_columnas": len(gdf.columns),
        "columnas_normalizadas": list(gdf.columns),
        "crs": str(gdf.crs),
        "epsg": gdf.crs.to_epsg(),
        "tipos_geometria": geometry_types,
        "geometrias_nulas": int(geometry.isna().sum()),
        "geometrias_vacias": empty_count,
        "geometrias_invalidas": invalid_count,
        "area_total_m2_aprox": area_total,
        "longitud_total_m_aprox": length_total,
    }


def clean_text_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    )


def _normalise_decimal_text(value: Any) -> Any:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().replace("\xa0", "").replace(" ", "")
    if text == "" or text.lower() in {"nan", "none", "<na>"}:
        return pd.NA
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    return text


def to_numeric_series(s: pd.Series) -> pd.Series:
    text = clean_text_series(s).map(_normalise_decimal_text)
    return pd.to_numeric(text, errors="coerce")


def normalize_label(value: Any) -> str | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = strip_accents(str(value)).strip().lower()
    return " ".join(text.split())


SER_COLOR_ALIASES = {
    "043000255 azul": "azul",
    "077214010 verde": "verde",
    "081209246 alta rotacion": "alta rotacion",
    "255000000 rojo": "rojo",
    "255140000 naranja": "naranja",
    "azul": "azul",
    "verde": "verde",
    "alta rotacion": "alta rotacion",
    "rojo": "rojo",
    "naranja": "naranja",
    "gris": "gris",
}


def normalize_ser_color(value: Any) -> str | pd.NA:
    label = normalize_label(value)
    if pd.isna(label):
        return pd.NA
    return SER_COLOR_ALIASES.get(label, label)


SER_REGULATED_COLORS_NORM = {
    "azul",
    "verde",
    "alta rotacion",
    "rojo",
    "naranja",
}


def compose_barrio_code(cod_distrito: pd.Series, num_barrio: pd.Series) -> pd.Series:
    cod_distrito_num = pd.to_numeric(cod_distrito, errors="coerce")
    num_barrio_num = pd.to_numeric(num_barrio, errors="coerce")
    return (cod_distrito_num * 100 + num_barrio_num).round().astype("Int64")


def union_geometry(gdf: gpd.GeoDataFrame):
    return gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union


def ensure_parquet_engine() -> None:
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError("Para escribir Parquet instala pyarrow en el entorno activo.") from exc


CARTOGRAPHY_COLUMN_ALIASES = {
    "ser_geoportal_limite_ser": {},
    "ser_geoportal_barrios_ser": {
        "coddis": "cod_distrito",
        "nomdis": "distrito",
        "codbar": "num_barrio",
        "nombar": "barrio",
    },
    "ser_geoportal_bandas_aparcamiento": {
        "id": "id_banda",
        "bateria_li": "bateria_linea",
        "res_numpla": "numero_plazas",
        "texto_caje": "texto_cajetin",
    },
}


def prepare_cartography_source(
    gdf: gpd.GeoDataFrame,
    dataset_id: str,
) -> gpd.GeoDataFrame:
    if dataset_id not in CARTOGRAPHY_COLUMN_ALIASES:
        raise ValueError(
            f"Dataset: {dataset_id}; no existe configuración en CARTOGRAPHY_COLUMN_ALIASES."
        )
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; se esperaba un GeoDataFrame; observado: {type(gdf).__name__}"
        )
    geometry_name = gdf.geometry.name
    if geometry_name not in gdf.columns:
        raise ValueError(f"Dataset: {dataset_id}; el GeoDataFrame no tiene geometría activa.")

    aliases = CARTOGRAPHY_COLUMN_ALIASES[dataset_id]
    rename_map = {source: target for source, target in aliases.items() if source in gdf.columns}
    for source, target in rename_map.items():
        if source != target and target in gdf.columns:
            raise ValueError(
                f"Dataset: {dataset_id}; columnas observadas: {list(gdf.columns)}; "
                f"renombrar {source!r} a {target!r} produciría una colisión."
            )

    renamed_columns = [rename_map.get(column, column) for column in gdf.columns]
    duplicated = pd.Index(renamed_columns)[pd.Index(renamed_columns).duplicated()].tolist()
    if duplicated:
        raise ValueError(
            f"Dataset: {dataset_id}; columnas observadas: {list(gdf.columns)}; "
            f"la operación produciría columnas duplicadas: {duplicated}"
        )

    prepared = gdf.rename(columns=rename_map).copy()
    geometry_after = rename_map.get(geometry_name, geometry_name)
    prepared = prepared.set_geometry(geometry_after)
    if geometry_after != "geometry":
        prepared = prepared.rename_geometry("geometry")
    return gpd.GeoDataFrame(prepared, geometry="geometry", crs=gdf.crs)


def require_columns(
    df: pd.DataFrame,
    required: list[str],
    dataset_id: str,
) -> None:
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(
            f"Dataset: {dataset_id}; columnas observadas: {list(df.columns)}; "
            f"columnas requeridas: {required}; columnas ausentes: {missing}"
        )


## 3. Inspección estructural individual de los raw

Cada fuente cartográfica se inspecciona antes de tomar decisiones de limpieza. No se presuponen esquemas: las columnas disponibles se observan después de leer cada raw y no se renombra ni deriva ninguna variable específica sin comprobar antes que existe.

La normalización realizada afecta únicamente a los nombres de columnas. No se muestran registros individuales ni geometrías. La inspección revisa estructura, CRS, tipos geométricos, geometrías nulas, geometrías vacías, geometrías inválidas y magnitudes espaciales aproximadas.

In [4]:
GEO_RAW: dict[str, gpd.GeoDataFrame] = {}
quality_rows = []

for dataset_id in CARTOGRAPHY_DATASET_IDS:
    raw_path = CARTOGRAPHY_RAW_PATHS[dataset_id]
    if not raw_path.exists():
        raise FileNotFoundError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: no existe; "
            "condición esperada: el raw candidato debe existir"
        )

    suffix = raw_path.suffix.lower()
    if suffix in {".geojson", ".json"}:
        raw_gdf = read_geojson(raw_path, dataset_id)
    elif suffix == ".zip":
        raw_gdf = read_shp_zip(raw_path, dataset_id)
    else:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: extensión {raw_path.suffix}; "
            "condición esperada: .geojson, .json o .zip"
        )

    observed_columns = list(raw_gdf.columns)
    if not observed_columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: 0 columnas; "
            "condición esperada: columnas estructurales observables"
        )

    normalized_gdf = normalize_geo_columns(raw_gdf)
    projected_gdf = ensure_crs_25830(normalized_gdf, dataset_id)

    if projected_gdf.empty:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: GeoDataFrame vacío; "
            "condición esperada: al menos una fila para inspección estructural"
        )

    GEO_RAW[dataset_id] = projected_gdf
    quality_rows.append(geometry_quality_summary(projected_gdf, dataset_id))

inspection_table = pd.DataFrame(quality_rows)

observed_geo_ids = list(GEO_RAW)
if observed_geo_ids != CARTOGRAPHY_DATASET_IDS:
    raise ValueError(
        "GEO_RAW no contiene exactamente los cuatro dataset_id cartográficos esperados. "
        f"Observado: {observed_geo_ids}; esperado: {CARTOGRAPHY_DATASET_IDS}"
    )

if len(inspection_table) != 4:
    raise ValueError(
        "La tabla de inspección debe contener exactamente cuatro filas cartográficas. "
        f"Valor observado: {len(inspection_table)}; esperado: 4"
    )

for dataset_id, gdf in GEO_RAW.items():
    observed_epsg = gdf.crs.to_epsg() if gdf.crs is not None else None
    if observed_epsg != 25830:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: EPSG {observed_epsg}; "
            "condición esperada: EPSG:25830"
        )

inspection_table

,dataset_id,n_filas,n_columnas,columnas_normalizadas,crs,epsg,tipos_geometria,geometrias_nulas,geometrias_vacias,geometrias_invalidas,area_total_m2_aprox,longitud_total_m_aprox
0,ser_geoportal_limite_ser,1,3,"[nombre, objectid, geometry]",EPSG:25830,25830,[Polygon],0,0,0,58686575.954764,<NA>
1,ser_geoportal_barrios_ser,67,6,"[coddis, nomdis, codbar, nombar, objectid, geometry]",EPSG:25830,25830,[Polygon],0,0,0,604455106.868571,<NA>
2,ser_geoportal_bandas_aparcamiento,87615,6,"[id, color, bateria_li, res_numpla, texto_caje, geometry]",EPSG:25830,25830,[LineString],0,0,0,<NA>,2185544.380116
3,callejero_viales_vigentes,9409,10,"[top_id, top_id_com, top_id_ter, tvia_id, top_dt_fch, top_dt_f_1, cv_tx_deno...",EPSG:25830,25830,"[MultiPolygon, Polygon]",122,0,37,89221620.640907,<NA>


### Lectura y criterio para continuar

La inspección estructural resume las cuatro fuentes cartográficas con el mismo criterio: lectura raw, normalización de nombres de columnas, CRS transformado a EPSG:25830 y recuento de geometrías nulas, vacías e inválidas.

El criterio de invalidez usado en la tabla es `~geometry.is_valid` sobre geometrías no nulas después de expresar cada fuente en EPSG:25830. Si el recuento observado para `callejero_viales_vigentes` difiere de diagnósticos previos, debe prevalecer este criterio reproducible dentro del notebook.

La lectura de esta tabla determina qué limpieza se aplica en cada bloque: validación y normalización para límite, filtrado semántico para barrios y bandas, y diagnóstico geométrico específico para la cartografía vial.

## 4. Limpieza de `ser_geoportal_limite_ser`

**Qué mide.** `ser_geoportal_limite_ser` contiene el polígono oficial del ámbito del Servicio de Estacionamiento Regulado.

**Uso en el TFM.** Sirve como geometría de referencia para validar si puntos o líneas SER caen dentro del área regulada y como base espacial para mapas posteriores.

**Columnas conservadas.** Se conservan `objectid`, `nombre` y `geometry`, porque identifican el polígono y su geometría oficial.

**Columnas descartadas.** No se añaden `dataset_id` ni `archivo_origen` al clean final, porque la fuente ya queda trazada por `data_catalog.csv` y por la ruta de salida.

**Validaciones.** Se comprueba número de geometrías, CRS, validez geométrica, geometrías nulas, área aproximada y nombres únicos. Esta fuente no requiere limpieza pesada: requiere validación y normalización geoespacial.


In [5]:
LIMITE_FINAL_COLUMNS = ["objectid", "nombre", "geometry"]

limite_input = prepare_cartography_source(
    GEO_RAW["ser_geoportal_limite_ser"],
    "ser_geoportal_limite_ser",
)
require_columns(
    limite_input,
    LIMITE_FINAL_COLUMNS,
    "ser_geoportal_limite_ser",
)


def clean_limite_ser(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    require_columns(gdf, LIMITE_FINAL_COLUMNS, "ser_geoportal_limite_ser")
    df = gdf.copy()
    out = gpd.GeoDataFrame({
        "objectid": pd.to_numeric(df["objectid"], errors="coerce").astype("Int64"),
        "nombre": clean_text_series(df["nombre"]),
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    return ensure_crs_25830(out, "ser_geoportal_limite_ser")


ser_geoportal_limite_ser_clean = clean_limite_ser(limite_input)
limite_geom_valid = ser_geoportal_limite_ser_clean.geometry.is_valid
limite_quality = pd.DataFrame([
    ("n_geometrias", int(len(ser_geoportal_limite_ser_clean)), "Número de geometrías del límite SER."),
    ("crs_epsg", int(ser_geoportal_limite_ser_clean.crs.to_epsg()), "Debe ser 25830 para medir distancias y áreas en metros."),
    ("n_geometrias_validas", int(limite_geom_valid.sum()), "Geometrías válidas."),
    ("n_geometrias_nulas", int(ser_geoportal_limite_ser_clean.geometry.isna().sum()), "Geometrías nulas."),
    ("area_aproximada_m2", float(ser_geoportal_limite_ser_clean.geometry.area.sum()), "Área total aproximada en m2."),
    ("nombres_unicos", ser_geoportal_limite_ser_clean["nombre"].dropna().unique().tolist(), "Nombres únicos de la capa."),
], columns=["check", "valor", "interpretacion"])

limite_quality

,check,valor,interpretacion
0,n_geometrias,1,Número de geometrías del límite SER.
1,crs_epsg,25830,Debe ser 25830 para medir distancias y áreas en metros.
2,n_geometrias_validas,1,Geometrías válidas.
3,n_geometrias_nulas,0,Geometrías nulas.
4,area_aproximada_m2,58686575.954764,Área total aproximada en m2.
5,nombres_unicos,[Zona S.E.R.],Nombres únicos de la capa.


**Lectura/decisión.** La capa contiene una única geometría válida, sin geometrías nulas, en EPSG:25830. El área aproximada es de 58.686.575,95 m² y el nombre único es `Zona S.E.R.`.

Con esta evidencia, el límite SER se acepta como geometría oficial de referencia. No se muestra una tabla limpia adicional porque el output final tiene una sola fila y su contenido ya queda suficientemente descrito por los checks.


## 5. Limpieza de `ser_geoportal_barrios_ser`

**Qué mide.** `ser_geoportal_barrios_ser` contiene los polígonos de barrios dentro del ámbito SER.

**Uso en el TFM.** Permite dividir el mapa SER por barrios y habilita futuras agregaciones espaciales por barrio. No sustituye a `ser_calles_plazas` como fuente de capacidad.

**Columnas conservadas.** Se conservan `cod_distrito`, `distrito`, `num_barrio`, `cod_barrio`, `barrio`, `objectid` y `geometry`. La regla `cod_barrio = cod_distrito * 100 + num_barrio` se usa como armonización del código compuesto de barrio.

**Columnas descartadas.** Se excluye del clean el polígono “No está en la zona SER”, porque no representa un barrio SER utilizable. También se descartan los flags de diagnóstico y la trazabilidad redundante, ya que solo se utilizan para justificar la selección de los polígonos que forman el output limpio.

**Validaciones.** Se comprueban conteos de polígonos, áreas, coherencia entre la unión de barrios y el límite SER, y duplicados de `cod_barrio`. Si aparece un duplicado, se diagnostica geométricamente antes de decidir si se elimina, se conserva o se pospone su disolución.


In [6]:
BARRIOS_INPUT_COLUMNS = [
    "cod_distrito",
    "distrito",
    "num_barrio",
    "barrio",
    "objectid",
    "geometry",
]

BARRIOS_FINAL_COLUMNS = [
    "cod_distrito",
    "distrito",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "objectid",
    "geometry",
]

TOL_AREA_M2 = 1.0

barrios_input = prepare_cartography_source(
    GEO_RAW["ser_geoportal_barrios_ser"],
    "ser_geoportal_barrios_ser",
)
require_columns(
    barrios_input,
    BARRIOS_INPUT_COLUMNS,
    "ser_geoportal_barrios_ser",
)


def clean_barrios_ser_diagnostic(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    require_columns(gdf, BARRIOS_INPUT_COLUMNS, "ser_geoportal_barrios_ser")
    df = gdf.copy()
    cod_distrito = pd.to_numeric(df["cod_distrito"], errors="coerce").astype("Int64")
    num_barrio = pd.to_numeric(df["num_barrio"], errors="coerce").astype("Int64")
    cod_barrio = compose_barrio_code(cod_distrito, num_barrio)
    barrio = clean_text_series(df["barrio"])
    barrio_norm = barrio.map(normalize_label)
    flag_en_zona_ser = barrio_norm.ne("no esta en la zona ser") & cod_barrio.notna()
    diagnostic = gpd.GeoDataFrame({
        "cod_distrito": cod_distrito,
        "distrito": clean_text_series(df["distrito"]),
        "num_barrio": num_barrio,
        "cod_barrio": cod_barrio,
        "barrio": barrio,
        "objectid": pd.to_numeric(df["objectid"], errors="coerce").astype("Int64"),
        "flag_en_zona_ser": flag_en_zona_ser.fillna(False),
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    return ensure_crs_25830(diagnostic, "ser_geoportal_barrios_ser")


def fmt_int(value: int) -> int:
    return int(value)


def fmt_m2(value: float) -> str:
    return f"{float(value):.3f}"


def fmt_pct(value: float) -> str:
    return f"{float(value):.9f}"


ser_geoportal_barrios_ser_diagnostic = clean_barrios_ser_diagnostic(barrios_input)
barrios_en_zona = ser_geoportal_barrios_ser_diagnostic["flag_en_zona_ser"]
barrios_candidate_clean = ser_geoportal_barrios_ser_diagnostic.loc[barrios_en_zona, BARRIOS_FINAL_COLUMNS].copy()

barrios_union_geom = union_geometry(barrios_candidate_clean)
limite_geom = union_geometry(ser_geoportal_limite_ser_clean)
area_union_barrios_ser_m2 = float(barrios_union_geom.area)
area_limite_ser_m2 = float(limite_geom.area)
diferencia_area_m2 = area_union_barrios_ser_m2 - area_limite_ser_m2
diferencia_area_pct = float(diferencia_area_m2 / area_limite_ser_m2 * 100) if area_limite_ser_m2 else float("nan")

barrios_quality = pd.DataFrame([
    ("n_poligonos_total", fmt_int(len(ser_geoportal_barrios_ser_diagnostic)), "Polígonos recibidos en raw."),
    ("n_poligonos_en_zona_ser", fmt_int(barrios_en_zona.sum()), "Polígonos SER reales que pasan a candidato clean."),
    ("n_poligonos_no_ser", fmt_int((~barrios_en_zona).sum()), "Polígonos no SER excluidos del clean."),
    ("area_total_m2", fmt_m2(ser_geoportal_barrios_ser_diagnostic.geometry.area.sum()), "Área total del raw."),
    ("area_en_zona_ser_m2", fmt_m2(ser_geoportal_barrios_ser_diagnostic.loc[barrios_en_zona].geometry.area.sum()), "Área sumada de polígonos SER conservados."),
    ("area_union_barrios_ser_m2", fmt_m2(area_union_barrios_ser_m2), "Área de la unión geométrica de barrios SER limpios."),
    ("area_limite_ser_m2", fmt_m2(area_limite_ser_m2), "Área del límite SER oficial."),
    ("diferencia_area_m2", fmt_m2(diferencia_area_m2), "Diferencia unión barrios SER menos límite SER."),
    ("diferencia_area_pct", fmt_pct(diferencia_area_pct), "Diferencia relativa sobre área del límite SER."),
    ("duplicados_cod_barrio_en_zona_ser", fmt_int(barrios_candidate_clean.duplicated("cod_barrio").sum()), "Duplicados de código compuesto dentro de zona SER."),
], columns=["check", "valor", "interpretacion"])

barrios_quality

,check,valor,interpretacion
0,n_poligonos_total,67,Polígonos recibidos en raw.
1,n_poligonos_en_zona_ser,66,Polígonos SER reales que pasan a candidato clean.
2,n_poligonos_no_ser,1,Polígonos no SER excluidos del clean.
3,area_total_m2,604455106.869,Área total del raw.
4,area_en_zona_ser_m2,58686575.971,Área sumada de polígonos SER conservados.
5,area_union_barrios_ser_m2,58686575.949,Área de la unión geométrica de barrios SER limpios.
6,area_limite_ser_m2,58686575.955,Área del límite SER oficial.
7,diferencia_area_m2,-0.005,Diferencia unión barrios SER menos límite SER.
8,diferencia_area_pct,-0.000000009,Diferencia relativa sobre área del límite SER.
9,duplicados_cod_barrio_en_zona_ser,1,Duplicados de código compuesto dentro de zona SER.


**Lectura/decisión.** El raw contiene 67 polígonos, de los cuales 66 corresponden a barrios SER reales y 1 corresponde a “No está en la zona SER”. Ese polígono no-SER se excluye del clean porque inflaría el área total y no debe intervenir en mapas ni agregaciones SER.

La unión geométrica de los barrios SER limpios tiene un área de 58.686.575,949 m², prácticamente idéntica al límite SER oficial, con una diferencia de -0,005 m² (-0,000000009 %). Esto valida que, tras excluir el polígono no-SER, la cobertura espacial de barrios SER cuadra con el límite oficial.

Se detecta un duplicado de `cod_barrio` dentro de zona SER, por lo que se realiza un diagnóstico específico antes de cerrar el clean.


### 5.1. Diagnóstico de duplicados de `cod_barrio`

El duplicado detectado corresponde al código `904`, asociado a `Valdezarza` y `Valdezarza Fase III`. Esta revisión comprueba si el duplicado implica un error geométrico real o si se trata de dos piezas territoriales con el mismo código compuesto.

Se comprueba: si las geometrías se tocan o intersectan, si existe solape relevante, si una contiene a la otra, si son disjuntas, y si la unión queda dentro del límite SER.


In [7]:
duplicados_barrios_ser = (
    barrios_candidate_clean
    .assign(area_m2=lambda df: df.geometry.area)
    .loc[lambda df: df.duplicated("cod_barrio", keep=False)]
    .sort_values(["cod_barrio", "barrio"])
)

if not duplicados_barrios_ser.empty:
    display(duplicados_barrios_ser.drop(columns="geometry"))

barrios_904 = barrios_candidate_clean.loc[barrios_candidate_clean["cod_barrio"].eq(904)].copy()
if len(barrios_904) == 2:
    geom_a, geom_b = barrios_904.geometry.iloc[0], barrios_904.geometry.iloc[1]
    union_904 = geom_a.union(geom_b)
    area_interseccion_m2 = float(geom_a.intersection(geom_b).area)
    cod_904_geometria = pd.DataFrame([{
        "cod_barrio": 904,
        "barrio_a": barrios_904["barrio"].iloc[0],
        "barrio_b": barrios_904["barrio"].iloc[1],
        "area_a_m2": float(geom_a.area),
        "area_b_m2": float(geom_b.area),
        "se_tocan_o_intersectan": bool(geom_a.intersects(geom_b)),
        "area_interseccion_m2": area_interseccion_m2,
        "solape_relevante": bool(area_interseccion_m2 > TOL_AREA_M2),
        "a_contiene_b": bool(geom_a.contains(geom_b)),
        "b_contiene_a": bool(geom_b.contains(geom_a)),
        "son_disjuntas": bool(geom_a.disjoint(geom_b)),
        "area_union_m2": float(union_904.area),
        "area_union_dentro_limite_ser_m2": float(union_904.intersection(limite_geom).area),
        "diferencia_union_vs_limite_intersec_m2": float(union_904.area - union_904.intersection(limite_geom).area),
    }])
elif len(barrios_904) > 0:
    cod_904_geometria = pd.DataFrame([{
        "cod_barrio": 904,
        "n_poligonos": int(len(barrios_904)),
        "nota": "No hay exactamente dos polígonos con cod_barrio 904; revisar manualmente si cambia el raw.",
    }])
else:
    cod_904_geometria = pd.DataFrame()

if not cod_904_geometria.empty:
    display(cod_904_geometria)


,cod_distrito,distrito,num_barrio,cod_barrio,barrio,objectid,area_m2
46,9,Moncloa - Aravaca,4,904,Valdezarza,47,359529.629332
65,9,Moncloa - Aravaca,4,904,Valdezarza Fase III,67,209555.978181


,cod_barrio,barrio_a,barrio_b,area_a_m2,area_b_m2,se_tocan_o_intersectan,area_interseccion_m2,solape_relevante,a_contiene_b,b_contiene_a,son_disjuntas,area_union_m2,area_union_dentro_limite_ser_m2,diferencia_union_vs_limite_intersec_m2
0,904,Valdezarza,Valdezarza Fase III,359529.629332,209555.978181,True,0.002087,False,False,False,False,569085.605426,569085.604794,0.000631


**Lectura/decisión.** El duplicado `904` tiene dos geometrías con áreas de 359.529,63 m² y 209.555,98 m². Aunque `se_tocan_o_intersectan = True`, el área de intersección es de solo 0,002087 m², por debajo de la tolerancia de 1 m²; por tanto, `solape_relevante = False`.

Ninguna geometría contiene a la otra y la diferencia entre el área de la unión y su intersección con el límite SER es residual. La decisión es conservar ambas geometrías en el clean, sin disolver ni eliminar. Si más adelante se necesita un único polígono por `cod_barrio`, la disolución se hará en un notebook posterior y deberá documentarse explícitamente.


In [8]:
ser_geoportal_barrios_ser_clean = barrios_candidate_clean.copy()

if list(ser_geoportal_barrios_ser_clean.columns) != BARRIOS_FINAL_COLUMNS:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; columnas finales inesperadas. "
        f"Observado: {list(ser_geoportal_barrios_ser_clean.columns)}; esperado: {BARRIOS_FINAL_COLUMNS}"
    )
if len(ser_geoportal_barrios_ser_clean) != 66:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; número de filas inesperado. "
        f"Observado: {len(ser_geoportal_barrios_ser_clean)}; esperado: 66"
    )
observed_epsg = ser_geoportal_barrios_ser_clean.crs.to_epsg() if ser_geoportal_barrios_ser_clean.crs is not None else None
if observed_epsg != 25830:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; EPSG inesperado. "
        f"Observado: {observed_epsg}; esperado: 25830"
    )
if ser_geoportal_barrios_ser_clean.geometry.isna().sum() != 0:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; geometrías nulas inesperadas. "
        f"Observado: {int(ser_geoportal_barrios_ser_clean.geometry.isna().sum())}; esperado: 0"
    )
invalid_count = int((~ser_geoportal_barrios_ser_clean.geometry.is_valid & ser_geoportal_barrios_ser_clean.geometry.notna()).sum())
if invalid_count != 0:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; geometrías inválidas inesperadas. "
        f"Observado: {invalid_count}; esperado: 0"
    )

## 6. Limpieza de `ser_geoportal_bandas_aparcamiento`

**Qué mide.** `ser_geoportal_bandas_aparcamiento` contiene la geometría lineal de las bandas de aparcamiento SER. Es la fuente que permite representar en un mapa las líneas coloreadas de plazas reguladas.

**Uso en el TFM.** Se mantiene como capa cartográfica de oferta física regulada y como contraste espacial frente a `ser_calles_plazas`. No es el denominador principal del modelo: la capacidad tabular histórica sigue viniendo de `ser_calles_plazas`.

**Columnas conservadas.** Se conservan `id_banda`, `color`, `numero_plazas` y `geometry`, porque identifican la banda, el tipo de plaza, el número de plazas y la línea que se dibujará.

**Columnas descartadas.** Se descartan `texto_cajetin`, `bateria_linea`, `longitud_m`, flags y trazabilidad de origen. `longitud_m` no se guarda porque puede recalcularse desde la geometría si se necesita en un análisis posterior.

**Validaciones.** Se comprueban colores regulados, bandas grises, plazas nulas/cero/negativas, geometrías inválidas y posición respecto al límite SER. La posición espacial se evalúa con criterio estricto y con un buffer de 5 m: el buffer evita sobrerreaccionar ante líneas situadas en el borde del polígono o desplazadas levemente por precisión cartográfica.

En esta limpieza interim no se eliminan bandas reguladas solo por quedar fuera del límite con buffer. Si tienen color SER válido, plazas informadas y geometría válida, se conservan y la incidencia espacial queda diagnosticada para el notebook posterior de mapas.


In [9]:
BANDAS_FINAL_COLUMNS = ["id_banda", "color", "numero_plazas", "geometry"]

bandas_input = prepare_cartography_source(
    GEO_RAW["ser_geoportal_bandas_aparcamiento"],
    "ser_geoportal_bandas_aparcamiento",
)
require_columns(
    bandas_input,
    [
        "id_banda",
        "color",
        "bateria_linea",
        "numero_plazas",
        "texto_cajetin",
        "geometry",
    ],
    "ser_geoportal_bandas_aparcamiento",
)


def diagnose_bandas_aparcamiento(gdf: gpd.GeoDataFrame, limite_geom) -> gpd.GeoDataFrame:
    require_columns(
        gdf,
        [
            "id_banda",
            "color",
            "bateria_linea",
            "numero_plazas",
            "texto_cajetin",
            "geometry",
        ],
        "ser_geoportal_bandas_aparcamiento",
    )
    df = gdf.copy()
    color_norm = clean_text_series(df["color"]).map(normalize_ser_color)
    numero_plazas = to_numeric_series(df["numero_plazas"]).round().astype("Int64")
    limite_geom_buffer_5m_local = limite_geom.buffer(5)
    diagnostic = gpd.GeoDataFrame({
        "id_banda": pd.to_numeric(df["id_banda"], errors="coerce").astype("Int64"),
        "color": color_norm.map(lambda x: str(x).replace(" ", "_") if pd.notna(x) else pd.NA).astype("string"),
        "color_norm_diagnostico": color_norm,
        "bateria_linea": clean_text_series(df["bateria_linea"]),
        "numero_plazas": numero_plazas,
        "texto_cajetin": clean_text_series(df["texto_cajetin"]),
        "longitud_m": df.geometry.length,
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    diagnostic = ensure_crs_25830(diagnostic, "ser_geoportal_bandas_aparcamiento")
    diagnostic["flag_color_gris"] = diagnostic["color_norm_diagnostico"].eq("gris").fillna(False)
    diagnostic["flag_color_ser_regulado"] = diagnostic["color_norm_diagnostico"].isin(SER_REGULATED_COLORS_NORM).fillna(False)
    diagnostic["flag_plazas_nulas"] = diagnostic["numero_plazas"].isna()
    diagnostic["flag_plazas_cero"] = diagnostic["numero_plazas"].fillna(-1).eq(0)
    diagnostic["flag_plazas_negativas"] = diagnostic["numero_plazas"].fillna(0).lt(0)
    diagnostic["flag_geom_invalida"] = ~diagnostic.geometry.is_valid | diagnostic.geometry.isna()
    diagnostic["flag_fuera_limite_estricto"] = ~diagnostic.geometry.intersects(limite_geom)
    diagnostic["flag_fuera_limite_buffer_5m"] = ~diagnostic.geometry.intersects(limite_geom_buffer_5m_local)
    return diagnostic


limite_geom = union_geometry(ser_geoportal_limite_ser_clean)
ser_geoportal_bandas_aparcamiento_diagnostic = diagnose_bandas_aparcamiento(
    bandas_input, limite_geom
)
bandas_diag = ser_geoportal_bandas_aparcamiento_diagnostic
bandas_keep = (
    bandas_diag["flag_color_ser_regulado"]
    & bandas_diag["numero_plazas"].notna()
    & ~bandas_diag["flag_geom_invalida"]
)
bandas_quality = pd.DataFrame([
    ("n_bandas_raw", int(len(bandas_diag)), "Bandas recibidas."),
    ("n_bandas_candidato_clean", int(bandas_keep.sum()), "Bandas SER reguladas válidas por color, plazas y geometría; el límite queda como diagnóstico."),
    ("n_bandas_sin_color", int(bandas_diag["color_norm_diagnostico"].isna().sum()), "Bandas sin color normalizable; se excluyen del clean."),
    ("n_bandas_gris", int(bandas_diag["flag_color_gris"].sum()), "Bandas grises diagnosticadas; se excluyen por no ser color SER regulado objetivo."),
    ("plazas_gris", int(bandas_diag.loc[bandas_diag["flag_color_gris"], "numero_plazas"].fillna(0).sum()), "Plazas asociadas a bandas grises."),
    ("n_plazas_nulas", int(bandas_diag["flag_plazas_nulas"].sum()), "Registros con numero_plazas nulo; se excluyen del clean."),
    ("n_plazas_cero", int(bandas_diag["flag_plazas_cero"].sum()), "Registros con cero plazas."),
    ("n_plazas_negativas", int(bandas_diag["flag_plazas_negativas"].sum()), "Registros con plazas negativas."),
    ("n_geometrias_invalidas", int(bandas_diag["flag_geom_invalida"].sum()), "Geometrías nulas o inválidas."),
    ("n_bandas_gris_fuera_limite_estricto", int((bandas_diag["flag_color_gris"] & bandas_diag["flag_fuera_limite_estricto"]).sum()), "Bandas grises fuera del límite estricto."),
    ("n_bandas_gris_fuera_limite_buffer_5m", int((bandas_diag["flag_color_gris"] & bandas_diag["flag_fuera_limite_buffer_5m"]).sum()), "Bandas grises fuera del límite con buffer 5 m."),
    ("n_bandas_reguladas_fuera_limite_estricto", int((bandas_diag["flag_color_ser_regulado"] & bandas_diag["flag_fuera_limite_estricto"]).sum()), "Bandas reguladas fuera del límite estricto."),
    ("n_bandas_reguladas_fuera_limite_buffer_5m", int((bandas_diag["flag_color_ser_regulado"] & bandas_diag["flag_fuera_limite_buffer_5m"]).sum()), "Bandas reguladas fuera incluso con buffer 5 m; se conservan como diagnóstico para el mapa."),
], columns=["check", "valor", "interpretacion"])

bandas_quality

,check,valor,interpretacion
0,n_bandas_raw,87615,Bandas recibidas.
1,n_bandas_candidato_clean,34450,"Bandas SER reguladas válidas por color, plazas y geometría; el límite queda ..."
2,n_bandas_sin_color,2,Bandas sin color normalizable; se excluyen del clean.
3,n_bandas_gris,53163,Bandas grises diagnosticadas; se excluyen por no ser color SER regulado obje...
4,plazas_gris,331380,Plazas asociadas a bandas grises.
5,n_plazas_nulas,2,Registros con numero_plazas nulo; se excluyen del clean.
6,n_plazas_cero,0,Registros con cero plazas.
7,n_plazas_negativas,0,Registros con plazas negativas.
8,n_geometrias_invalidas,0,Geometrías nulas o inválidas.
9,n_bandas_gris_fuera_limite_estricto,53151,Bandas grises fuera del límite estricto.


**Lectura/decisión.** El clean conserva las bandas con color SER regulado, `numero_plazas` informado y geometría válida. Se excluyen las bandas sin color normalizable y los registros con `numero_plazas` nulo, porque no pueden simbolizarse ni aportar capacidad fiable en la capa cartográfica.

Las bandas grises se excluyen porque no pertenecen a los colores SER regulados objetivo del TFM (`azul`, `verde`, `alta_rotacion`, `rojo`, `naranja`). No se eliminan porque todas estén fuera del límite: una parte queda dentro o cerca del ámbito SER. La justificación correcta es semántica/cartográfica, no puramente espacial.

En cuanto al límite SER, el notebook distingue entre fuera del límite estricto y fuera con buffer de 5 m. Las bandas reguladas que siguen fuera incluso con buffer se mantienen en el clean interim porque tienen color SER válido y geometría válida. Su tratamiento cartográfico se decidirá al construir el mapa, donde podrán excluirse únicamente de una representación concreta si la inspección visual demuestra que son incoherentes. Esta decisión no modificará el output limpio.


In [10]:
ser_geoportal_bandas_aparcamiento_clean = bandas_diag.loc[
    bandas_keep,
    BANDAS_FINAL_COLUMNS,
].copy()

if list(ser_geoportal_bandas_aparcamiento_clean.columns) != BANDAS_FINAL_COLUMNS:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; columnas finales inesperadas. "
        f"Observado: {list(ser_geoportal_bandas_aparcamiento_clean.columns)}; esperado: {BANDAS_FINAL_COLUMNS}"
    )
if len(ser_geoportal_bandas_aparcamiento_clean) != 34450:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; número de filas inesperado. "
        f"Observado: {len(ser_geoportal_bandas_aparcamiento_clean)}; esperado: 34450"
    )
observed_epsg = ser_geoportal_bandas_aparcamiento_clean.crs.to_epsg() if ser_geoportal_bandas_aparcamiento_clean.crs is not None else None
if observed_epsg != 25830:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; EPSG inesperado. "
        f"Observado: {observed_epsg}; esperado: 25830"
    )
if ser_geoportal_bandas_aparcamiento_clean.geometry.isna().sum() != 0:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; geometrías nulas inesperadas. "
        f"Observado: {int(ser_geoportal_bandas_aparcamiento_clean.geometry.isna().sum())}; esperado: 0"
    )
invalid_count = int((~ser_geoportal_bandas_aparcamiento_clean.geometry.is_valid & ser_geoportal_bandas_aparcamiento_clean.geometry.notna()).sum())
if invalid_count != 0:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; geometrías inválidas inesperadas. "
        f"Observado: {invalid_count}; esperado: 0"
    )
if ser_geoportal_bandas_aparcamiento_clean["numero_plazas"].isna().sum() != 0:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; numero_plazas nulos inesperados. "
        f"Observado: {int(ser_geoportal_bandas_aparcamiento_clean['numero_plazas'].isna().sum())}; esperado: 0"
    )
allowed_clean_colors = {"azul", "verde", "alta_rotacion", "rojo", "naranja"}
observed_colors = set(ser_geoportal_bandas_aparcamiento_clean["color"].dropna().astype(str).unique())
unexpected_colors = sorted(observed_colors - allowed_clean_colors)
if unexpected_colors:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; colores finales fuera de catálogo. "
        f"Observado: {unexpected_colors}; esperado: {sorted(allowed_clean_colors)}"
    )

## 7. Limpieza de `callejero_viales_vigentes`

**Qué mide.** `callejero_viales_vigentes` contiene cartografía vial vigente del municipio. En este notebook se usa como base vial neutra para contextualizar la representación de las bandas SER.

**Uso en el TFM.** El callejero actúa como fondo vial poligonal y como capa de identificación de calles. Las bandas SER siguen siendo la fuente que representa los lados y colores del estacionamiento; el callejero no reemplaza a las bandas, sino que las contextualiza espacialmente.

**Columnas conservadas.** Se conserva `top_id` como identificador técnico del vial. Se conservan `cv_tx_deno`, `den_tx_par` y `den_tx_nom` cuando existen tras normalización, porque permiten reconstruir el nombre de la calle. Se deriva `nombre_via_completo` para etiquetado, auditoría visual y posibles joins espaciales posteriores con bandas SER. Se conserva `geometry` como superficie vial.

**Columnas descartadas.** Se descartan atributos no necesarios para un fondo vial neutro, campos auxiliares y columnas sin papel directo en identificación, etiquetado o representación geométrica.

**Validaciones.** Se diagnostican geometrías nulas, vacías e inválidas, tipos geométricos, área o longitud aproximada, cobertura respecto al límite SER, cobertura de `top_id` y cobertura de `nombre_via_completo`. Las geometrías nulas se excluyen porque no pueden representarse. Las geometrías inválidas solo se reparan cuando `make_valid` está disponible, mejora la validez, no genera geometrías vacías y no cambia de forma desproporcionada el área total.

**Diagnóstico y decisión.** La decisión de limpieza se basa en la tabla de diagnóstico del bloque. Si falta algún componente de nombre (`cv_tx_deno`, `den_tx_par` o `den_tx_nom`), la tabla lo documenta y `nombre_via_completo` se construye con las partes disponibles. Si la cobertura del nombre completo es parcial, se conserva como limitación; si es nula, el notebook se detiene porque el callejero no aportaría identificación de calles.

In [11]:
CALLEJERO_DATASET_ID = "callejero_viales_vigentes"
CALLEJERO_REPAIR_MAX_AREA_DELTA_PCT = 0.01
CALLEJERO_ALLOWED_GEOMETRY_TYPES = {"Polygon", "MultiPolygon", "LineString", "MultiLineString"}
CALLEJERO_BASE_COLUMNS = ["top_id", "cv_tx_deno", "den_tx_par", "den_tx_nom"]
CALLEJERO_NAME_COMPONENTS = ["cv_tx_deno", "den_tx_par", "den_tx_nom"]


def build_nombre_via_completo(df: pd.DataFrame, component_columns: list[str]) -> pd.Series:
    if not component_columns:
        return pd.Series(pd.NA, index=df.index, dtype="string")
    parts = df[component_columns].apply(lambda col: clean_text_series(col))
    combined = parts.apply(
        lambda row: " ".join(str(value) for value in row if pd.notna(value) and str(value).strip()),
        axis=1,
    )
    return clean_text_series(combined)


callejero_raw = GEO_RAW[CALLEJERO_DATASET_ID].copy()
callejero_geom = callejero_raw.geometry
callejero_non_null_geom = callejero_geom[callejero_geom.notna()]
callejero_invalid_mask = callejero_geom.notna() & ~callejero_geom.is_valid
callejero_empty_mask = callejero_geom.notna() & callejero_geom.is_empty
callejero_type_counts = callejero_non_null_geom.geom_type.value_counts(dropna=False).to_dict()

limite_geom = union_geometry(ser_geoportal_limite_ser_clean)

callejero_area_before = float(
    callejero_non_null_geom.loc[
        callejero_non_null_geom.geom_type.isin(["Polygon", "MultiPolygon"])
    ].area.sum()
)
callejero_length_before = float(
    callejero_non_null_geom.loc[
        callejero_non_null_geom.geom_type.isin(["LineString", "MultiLineString"])
    ].length.sum()
)

callejero_repair_applied = False
callejero_repair_reason = "sin geometrías inválidas"
callejero_repaired = callejero_raw.copy()
callejero_invalid_before = int(callejero_invalid_mask.sum())
callejero_invalid_after = callejero_invalid_before
callejero_empty_after = int(callejero_empty_mask.sum())
callejero_area_after = callejero_area_before
callejero_area_delta_pct = 0.0 if callejero_area_before else pd.NA

if callejero_invalid_before:
    if shapely_make_valid is None:
        callejero_repair_reason = "make_valid no disponible; se excluyen geometrías inválidas"
    else:
        repaired_geometry = callejero_geom.copy()
        repaired_geometry.loc[callejero_invalid_mask] = repaired_geometry.loc[callejero_invalid_mask].map(shapely_make_valid)
        repaired_non_null = repaired_geometry[repaired_geometry.notna()]
        repaired_invalid_count = int((~repaired_non_null.is_valid).sum())
        repaired_empty_count = int(repaired_non_null.is_empty.sum())
        repaired_area = float(
            repaired_non_null.loc[
                repaired_non_null.geom_type.isin(["Polygon", "MultiPolygon"])
            ].area.sum()
        )
        area_delta_pct = (
            abs(repaired_area - callejero_area_before) / callejero_area_before * 100
            if callejero_area_before
            else pd.NA
        )
        repair_ok = (
            repaired_invalid_count < callejero_invalid_before
            and repaired_empty_count == 0
            and (pd.isna(area_delta_pct) or area_delta_pct <= CALLEJERO_REPAIR_MAX_AREA_DELTA_PCT)
        )
        if repair_ok:
            callejero_repaired = callejero_raw.copy()
            callejero_repaired["geometry"] = repaired_geometry
            callejero_repaired = gpd.GeoDataFrame(callejero_repaired, geometry="geometry", crs=callejero_raw.crs)
            callejero_repair_applied = True
            callejero_repair_reason = "make_valid aceptado"
            callejero_invalid_after = repaired_invalid_count
            callejero_empty_after = repaired_empty_count
            callejero_area_after = repaired_area
            callejero_area_delta_pct = area_delta_pct
        else:
            callejero_repair_reason = "make_valid no cumple los criterios; se excluyen geometrías inválidas"
            callejero_invalid_after = repaired_invalid_count
            callejero_empty_after = repaired_empty_count
            callejero_area_after = repaired_area
            callejero_area_delta_pct = area_delta_pct

callejero_existing_base_columns = [col for col in CALLEJERO_BASE_COLUMNS if col in callejero_repaired.columns]
callejero_missing_base_columns = [col for col in CALLEJERO_BASE_COLUMNS if col not in callejero_repaired.columns]
callejero_existing_name_components = [col for col in CALLEJERO_NAME_COMPONENTS if col in callejero_repaired.columns]

if "top_id" not in callejero_repaired.columns:
    raise ValueError(
        "Dataset: callejero_viales_vigentes; falta la columna top_id tras normalizar. "
        f"Columnas observadas: {list(callejero_repaired.columns)}"
    )
if "den_tx_nom" not in callejero_repaired.columns:
    raise ValueError(
        "Dataset: callejero_viales_vigentes; falta den_tx_nom o equivalente normalizado. "
        f"Columnas observadas: {list(callejero_repaired.columns)}"
    )

callejero_repaired = callejero_repaired.copy()
callejero_repaired["nombre_via_completo"] = build_nombre_via_completo(
    callejero_repaired,
    callejero_existing_name_components,
)

callejero_clean_mask = (
    callejero_repaired.geometry.notna()
    & ~callejero_repaired.geometry.is_empty
    & callejero_repaired.geometry.is_valid
    & callejero_repaired.geometry.geom_type.isin(CALLEJERO_ALLOWED_GEOMETRY_TYPES)
)
callejero_final_columns = [
    *callejero_existing_base_columns,
    "nombre_via_completo",
    "geometry",
]
callejero_viales_vigentes_clean = callejero_repaired.loc[
    callejero_clean_mask,
    callejero_final_columns,
].copy()
callejero_viales_vigentes_clean = ensure_crs_25830(callejero_viales_vigentes_clean, CALLEJERO_DATASET_ID)

if callejero_viales_vigentes_clean.empty:
    raise ValueError("Dataset: callejero_viales_vigentes; el clean queda vacío tras aplicar criterios geométricos.")
if callejero_viales_vigentes_clean.geometry.isna().any():
    raise ValueError("Dataset: callejero_viales_vigentes; el clean conserva geometrías nulas.")
if callejero_viales_vigentes_clean.geometry.is_empty.any():
    raise ValueError("Dataset: callejero_viales_vigentes; el clean conserva geometrías vacías.")
invalid_clean = int((~callejero_viales_vigentes_clean.geometry.is_valid).sum())
if invalid_clean:
    raise ValueError(
        "Dataset: callejero_viales_vigentes; el clean conserva geometrías inválidas. "
        f"Valor observado: {invalid_clean}; esperado: 0"
    )
unexpected_clean_types = sorted(set(callejero_viales_vigentes_clean.geometry.geom_type.unique()) - CALLEJERO_ALLOWED_GEOMETRY_TYPES)
if unexpected_clean_types:
    raise ValueError(
        "Dataset: callejero_viales_vigentes; tipos geométricos no admitidos en el clean. "
        f"Observado: {unexpected_clean_types}; esperado: {sorted(CALLEJERO_ALLOWED_GEOMETRY_TYPES)}"
    )
if callejero_viales_vigentes_clean["top_id"].isna().all():
    raise ValueError("Dataset: callejero_viales_vigentes; top_id está completamente nulo en el clean.")
if "nombre_via_completo" not in callejero_viales_vigentes_clean.columns:
    raise ValueError("Dataset: callejero_viales_vigentes; falta nombre_via_completo en el clean.")

n_nombre_via_completo_nulos = int(callejero_viales_vigentes_clean["nombre_via_completo"].isna().sum())
cobertura_nombre_via_completo_pct = (
    (1 - n_nombre_via_completo_nulos / len(callejero_viales_vigentes_clean)) * 100
    if len(callejero_viales_vigentes_clean)
    else 0.0
)
if cobertura_nombre_via_completo_pct == 0:
    raise ValueError(
        "Dataset: callejero_viales_vigentes; nombre_via_completo tiene cobertura 0 %. "
        "La fuente no aporta nombres de calle para el mapa."
    )

n_top_id_nulos = int(callejero_viales_vigentes_clean["top_id"].isna().sum())

callejero_clean_intersects_limite = (
    callejero_viales_vigentes_clean.geometry
    .intersects(limite_geom)
    .fillna(False)
)

callejero_clean_intersects_buffer_5m = (
    callejero_viales_vigentes_clean.geometry
    .intersects(limite_geom.buffer(5))
    .fillna(False)
)

callejero_quality = pd.DataFrame([
    ("n_filas_raw", int(len(callejero_raw)), "Registros recibidos en raw."),
    ("columnas_conservadas", callejero_final_columns, "Columnas que pasan al clean."),
    ("columnas_nombre_faltantes", callejero_missing_base_columns, "Componentes de identificación o nombre no presentes tras normalizar."),
    ("n_top_id_nulos", n_top_id_nulos, "Registros clean sin identificador técnico top_id."),
    ("n_nombre_via_completo_nulos", n_nombre_via_completo_nulos, "Registros clean sin nombre de vía derivado."),
    ("cobertura_nombre_via_completo_pct", round(cobertura_nombre_via_completo_pct, 3), "Porcentaje de registros clean con nombre de vía derivado."),
    ("tipos_geometria_raw", callejero_type_counts, "Tipos geométricos observados antes de limpiar."),
    ("n_geometrias_nulas", int(callejero_geom.isna().sum()), "Geometrías nulas excluidas del clean."),
    ("n_geometrias_vacias", int(callejero_empty_mask.sum()), "Geometrías vacías excluidas del clean."),
    ("n_geometrias_invalidas_antes", callejero_invalid_before, "Geometrías inválidas antes de reparar o excluir."),
    ("make_valid_disponible", bool(shapely_make_valid is not None), "Disponibilidad de reparación geométrica."),
    ("reparacion_aplicada", bool(callejero_repair_applied), callejero_repair_reason),
    ("n_geometrias_invalidas_despues_reparacion", callejero_invalid_after, "Geometrías inválidas tras evaluar reparación."),
    ("area_total_m2_antes", callejero_area_before, "Área aproximada antes de limpiar."),
    ("delta_area_pct_reparacion", callejero_area_delta_pct, "Cambio relativo de área por reparación."),
    ("n_clean_intersecta_limite_ser", int(callejero_clean_intersects_limite.sum()), "Geometrías limpias que intersectan el límite SER."),
    ("n_clean_intersecta_limite_ser_buffer_5m", int(callejero_clean_intersects_buffer_5m.sum()), "Geometrías limpias que intersectan el límite SER con buffer 5 m."),
    ("n_filas_clean", int(len(callejero_viales_vigentes_clean)), "Registros conservados en el clean."),
], columns=["check", "valor", "interpretacion"])

callejero_quality

,check,valor,interpretacion
0,n_filas_raw,9409,Registros recibidos en raw.
1,columnas_conservadas,"[top_id, cv_tx_deno, den_tx_par, den_tx_nom, nombre_via_completo, geometry]",Columnas que pasan al clean.
2,columnas_nombre_faltantes,[],Componentes de identificación o nombre no presentes tras normalizar.
3,n_top_id_nulos,0,Registros clean sin identificador técnico top_id.
4,n_nombre_via_completo_nulos,0,Registros clean sin nombre de vía derivado.
5,cobertura_nombre_via_completo_pct,100.0,Porcentaje de registros clean con nombre de vía derivado.
6,tipos_geometria_raw,"{'Polygon': 8639, 'MultiPolygon': 648}",Tipos geométricos observados antes de limpiar.
7,n_geometrias_nulas,122,Geometrías nulas excluidas del clean.
8,n_geometrias_vacias,0,Geometrías vacías excluidas del clean.
9,n_geometrias_invalidas_antes,37,Geometrías inválidas antes de reparar o excluir.


**Lectura/decisión.** El raw contiene 9.409 registros, de los cuales se conservan 9.287. Las 122 geometrías nulas se excluyen porque no pueden representarse ni participar en operaciones espaciales. Las 37 geometrías inválidas se reparan mediante `make_valid`, ya que la reparación reduce la invalidez a cero y no modifica el área total de forma apreciable (`delta_area_pct_reparacion = 0.0`). Por tanto, la reparación se acepta como una corrección geométrica controlada.

La capa limpia contiene geometrías poligonales (`Polygon` y `MultiPolygon`), por lo que debe interpretarse como superficie vial, no como eje lineal de calle. De las 9.287 geometrías limpias, 3.266 intersectan directamente el límite SER y 3.268 lo hacen con un buffer de 5 m. Esta cobertura se utiliza solo como diagnóstico: no se filtran aquí las geometrías fuera del SER porque el callejero representa cartografía vial municipal completa. El recorte al ámbito SER, si procede, debe realizarse posteriormente en la fase de visualización mediante intersección, clip o buffer.

En consecuencia, `callejero_viales_vigentes_clean` se acepta como capa cartográfica limpia para fondo vial e identificación de calles. Las bandas SER siguen siendo la fuente encargada de representar los lados, colores y plazas del estacionamiento regulado; el callejero aporta el contexto vial necesario para interpretar esas bandas en el mapa.


## 8. Escritura de salidas limpias

Se escriben las cuatro salidas cartográficas limpias en `data/interim/cartografia/`: límite SER, barrios SER, bandas SER y callejero/viales vigentes.

La escritura no genera mapas, joins, agregados ni archivos `processed`. La tabla final muestra solo la ruta escrita, la dimensión de cada salida y el estado técnico de escritura.

In [12]:
ensure_parquet_engine()

cartography_clean_outputs = {
    "ser_geoportal_limite_ser": ser_geoportal_limite_ser_clean,
    "ser_geoportal_barrios_ser": ser_geoportal_barrios_ser_clean,
    "ser_geoportal_bandas_aparcamiento": ser_geoportal_bandas_aparcamiento_clean,
    "callejero_viales_vigentes": callejero_viales_vigentes_clean,
}

missing_outputs = [
    dataset_id for dataset_id in CARTOGRAPHY_DATASET_IDS
    if dataset_id not in cartography_clean_outputs
]
if missing_outputs:
    raise ValueError(f"Faltan salidas limpias para escribir: {missing_outputs}")

write_rows = []
for dataset_id in CARTOGRAPHY_DATASET_IDS:
    clean_gdf = cartography_clean_outputs[dataset_id]
    if clean_gdf.empty:
        raise ValueError(f"Dataset: {dataset_id}; la salida limpia está vacía y no se escribe.")
    output_path = CANDIDATE_INTERIM_PATHS[dataset_id]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    clean_gdf.to_parquet(output_path, index=False)
    if not output_path.exists():
        raise FileNotFoundError(
            f"Dataset: {dataset_id}; ruta: {relpath(output_path)}; no se localiza el Parquet escrito."
        )
    write_rows.append({
        "dataset_id": dataset_id,
        "archivo_interim": relpath(output_path),
        "shape_escrita": clean_gdf.shape,
        "size_mb": round(output_path.stat().st_size / 1024 / 1024, 3),
        "estado_escritura": "OK",
    })

write_check = pd.DataFrame(write_rows)
write_check

,dataset_id,archivo_interim,shape_escrita,size_mb,estado_escritura
0,ser_geoportal_limite_ser,data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_c...,"(1, 3)",0.043,OK
1,ser_geoportal_barrios_ser,data/interim/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser...,"(66, 7)",0.081,OK
2,ser_geoportal_bandas_aparcamiento,data/interim/cartografia/ser_geoportal_bandas_aparcamiento/ser_geoportal_ban...,"(34450, 4)",1.273,OK
3,callejero_viales_vigentes,data/interim/cartografia/callejero_viales_vigentes/callejero_viales_vigentes...,"(9287, 6)",17.561,OK


### Lectura final

El notebook deja preparadas cuatro capas cartográficas limpias: el límite SER, los barrios SER, las bandas SER y el callejero/viales vigentes. El límite aporta la geometría oficial del ámbito regulado; los barrios conservan los polígonos útiles y las dos piezas territoriales del código 904; las bandas mantienen las líneas reguladas válidas por color, plazas y geometría; y el callejero conserva identificador técnico, componentes del nombre de vía, nombre completo derivado y geometría.

Todas las salidas se escriben en `data/interim/cartografia/`. No se generan mapas, joins, agregados ni métricas de dificultad.